# Datathon VinU - Multi GPU Modeling (Optimized)

Notebook standalone (1 file duy nh?t) cho pipeline t?i ?u.


In [ ]:
# Kaggle setup (run once/session)
!pip -q install --upgrade xgboost optuna holidays prophet shap joblib tqdm


In [ ]:
"""Optimized forecasting pipeline for Kaggle notebook and local runs.

Upgrades over the legacy notebook:
- Leakage-safe external feature store from multiple tables.
- Target decomposition with COGS ratio branch.
- Horizon-bucket blend weights.
- Optional Optuna tuning.
- Multi-seed bagging for final models.
- SHAP export and richer metrics (MASE/sMAPE).
"""

from __future__ import annotations

import gc
import json
import logging
import warnings
from pathlib import Path
from typing import Dict, Iterator, List, Optional, Tuple

import holidays
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from joblib import Parallel, delayed
from prophet import Prophet
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX

try:
    from scipy.optimize import minimize
except Exception:
    minimize = None

warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").setLevel(logging.CRITICAL)
logging.getLogger("prophet").setLevel(logging.CRITICAL)

SEED = 42
DATE_COL = "date"

EVENT_COLS = [
    "is_tet_week",
    "is_dd_9_9",
    "is_dd_10_10",
    "is_dd_11_11",
    "is_dd_12_12",
    "is_black_friday",
    "is_payday_window",
]

LAGS = [1, 2, 3, 7, 14, 21, 28, 30, 60, 90, 180, 365]
ROLL_WINDOWS = [7, 14, 28, 56, 90]
EWM_ALPHAS = [0.05, 0.1, 0.2, 0.4]
FORECAST_HISTORY_WINDOW = 500

TET_DATES = pd.to_datetime(
    [
        "2012-01-23",
        "2013-02-10",
        "2014-01-31",
        "2015-02-19",
        "2016-02-08",
        "2017-01-28",
        "2018-02-16",
        "2019-02-05",
        "2020-01-25",
        "2021-02-12",
        "2022-02-01",
        "2023-01-22",
        "2024-02-10",
        "2025-01-29",
        "2026-02-17",
        "2027-02-06",
        "2028-01-26",
    ]
)

HORIZON_BUCKETS = [
    (1, 30, "h01_030"),
    (31, 90, "h031_090"),
    (91, 180, "h091_180"),
    (181, 365, "h181_365"),
    (366, 9999, "h366_plus"),
]


class Config:
    def __init__(self) -> None:
        self.show_progress = True
        self.use_cross_table_features = True
        self.train_recent_years: Optional[int] = None

        self.cv_max_splits = 6
        self.initial_train_days = 365 * 5
        self.val_days = 548
        self.step_days = 150
        self.gap_days = 28

        self.enable_optuna = True
        self.n_trials = 80
        self.optuna_timeout = 3600

        self.use_prophet_cv = False
        self.use_prophet_final = False
        self.use_residual_sarima = True
        self.use_residual_stacker = True
        self.use_huber_model = True
        self.huber_slope = 5e5

        self.use_multi_seed = True
        self.bagging_seeds = [42, 142, 242, 342, 442]

        self.xgb_cv_estimators = 2500
        self.xgb_final_estimators = 2200
        self.xgb_n_jobs = 1
        self.lam_blend = 0.01
        self.composite_alpha = 0.5
        self.dirichlet_prior_strength = 0.02

        self.use_duan_smearing = True
        self.use_linear_calibration = True
        self.use_prediction_winsor = True
        self.winsor_q_low = 0.01
        self.winsor_q_high = 0.99
        self.growth_headroom_annual = 0.15
        self.lower_clip_scale = 0.5
        self.upper_clip_scale = 1.5

        self.force_sequential = True


CFG = Config()


def _resolve_file_case_insensitive(root: Path, filename: str) -> Optional[Path]:
    if (not root.exists()) or (not root.is_dir()):
        return None
    target = filename.lower()
    for x in root.iterdir():
        if x.is_file() and x.name.lower() == target:
            return x
    return None


def _has_required_files(folder: Path) -> bool:
    return (
        _resolve_file_case_insensitive(folder, "sales.csv") is not None
        and _resolve_file_case_insensitive(folder, "sample_submission.csv") is not None
    )


def find_data_dir(explicit: Optional[Path] = None) -> Path:
    def _scan_folder(folder: Path) -> Optional[Path]:
        if (not folder.exists()) or (not folder.is_dir()):
            return None
        if _has_required_files(folder):
            return folder
        # one-level nested fallback
        try:
            for child in folder.iterdir():
                if child.is_dir() and _has_required_files(child):
                    return child
        except Exception:
            pass
        return None

    if explicit is not None:
        explicit = Path(explicit)
        if explicit.is_file():
            explicit = explicit.parent
        hit = _scan_folder(explicit)
        if hit is not None:
            return hit
        # deep scan under explicit root
        for d in explicit.rglob("*"):
            if d.is_dir() and _has_required_files(d):
                return d

    candidates = [
        Path("/kaggle/input/datathon-vinu/dataset"),
        Path("/kaggle/input/datathon_vinu/dataset"),
        Path("/kaggle/input/datathonvinu/dataset"),
        Path("/kaggle/input/DatathonVinU/dataset"),
        Path("/kaggle/input/datathon-vinu"),
        Path("/kaggle/input/datathon_vinu"),
        Path("/kaggle/input/datathonvinu"),
        Path("/kaggle/input/DatathonVinU"),
        Path("/kaggle/working/dataset"),
        Path("./dataset"),
    ]

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for ds in sorted(kaggle_input.iterdir()):
            if ds.is_dir():
                candidates.extend([ds, ds / "dataset"])

    seen = set()
    for p in candidates:
        p = Path(p)
        if (not p.exists()) or (not p.is_dir()):
            continue
        key = str(p.resolve())
        if key in seen:
            continue
        seen.add(key)
        hit = _scan_folder(p)
        if hit is not None:
            return hit

    # recursive fallback across common roots
    search_roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
    for root in search_roots:
        if (not root.exists()) or (not root.is_dir()):
            continue
        for sales_fp in root.rglob("*"):
            if not sales_fp.is_file():
                continue
            if sales_fp.name.lower() != "sales.csv":
                continue
            parent = sales_fp.parent
            if _resolve_file_case_insensitive(parent, "sample_submission.csv") is not None:
                return parent
            # sibling folder fallback
            try:
                for sib in parent.parent.iterdir():
                    if sib.is_dir() and _has_required_files(sib):
                        return sib
            except Exception:
                pass

    # helpful debug message
    debug_lines = []
    if kaggle_input.exists():
        debug_lines.append("Available /kaggle/input dirs:")
        for ds in sorted(kaggle_input.iterdir()):
            if ds.is_dir():
                debug_lines.append(f"- {ds}")
    msg = "Khong tim thay sales.csv + sample_submission.csv."
    if debug_lines:
        msg += "\n" + "\n".join(debug_lines)
    raise FileNotFoundError(msg)

def read_csv_if_exists(path: Path, parse_dates: Optional[List[str]] = None, usecols: Optional[List[str]] = None) -> Optional[pd.DataFrame]:
    if not path.exists():
        return None
    try:
        return pd.read_csv(path, parse_dates=parse_dates, usecols=usecols)
    except Exception:
        try:
            return pd.read_csv(path, parse_dates=parse_dates)
        except Exception:
            return None


def add_basic_calendar(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    d = out[DATE_COL].dt
    out["day_of_week"] = d.dayofweek
    out["day_of_month"] = d.day
    out["day_of_year"] = d.dayofyear
    out["week_of_year"] = d.isocalendar().week.astype(int)
    out["month"] = d.month
    out["quarter"] = d.quarter
    out["year"] = d.year
    out["is_weekend"] = (d.dayofweek >= 5).astype("int8")
    out["is_month_start"] = d.is_month_start.astype("int8")
    out["is_month_end"] = d.is_month_end.astype("int8")
    out["is_quarter_start"] = d.is_quarter_start.astype("int8")
    out["is_quarter_end"] = d.is_quarter_end.astype("int8")
    out["is_year_start"] = d.is_year_start.astype("int8")
    out["is_year_end"] = d.is_year_end.astype("int8")
    out["time_idx"] = np.arange(len(out), dtype=np.int32)
    return out


def add_cyclical(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col, period in [("day_of_week", 7), ("month", 12), ("day_of_year", 365.25), ("day_of_month", 31)]:
        out[f"{col}_sin"] = np.sin(2 * np.pi * out[col] / period)
        out[f"{col}_cos"] = np.cos(2 * np.pi * out[col] / period)
    return out


def add_tet_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    d = out[DATE_COL].values.astype("datetime64[D]")
    tet = TET_DATES.values.astype("datetime64[D]")
    nxt = np.clip(np.searchsorted(tet, d, side="left"), 0, len(tet) - 1)
    prv = np.clip(nxt - 1, 0, len(tet) - 1)
    days_to = (tet[nxt] - d).astype(int)
    days_since = (d - tet[prv]).astype(int)
    out["days_to_tet"] = days_to
    out["days_since_tet"] = days_since
    out["is_tet_day"] = (days_to == 0).astype("int8")
    out["is_tet_week"] = ((days_to <= 3) | (days_since <= 3)).astype("int8")
    out["is_pre_tet_14d"] = ((days_to >= 1) & (days_to <= 14)).astype("int8")
    out["is_pre_tet_30d"] = ((days_to >= 1) & (days_to <= 30)).astype("int8")
    out["tet_proximity"] = np.exp(-np.minimum(np.abs(days_to), np.abs(days_since)) / 7.0)
    return out


def _black_friday(year: int) -> pd.Timestamp:
    nov = pd.date_range(f"{year}-11-01", f"{year}-11-30", freq="D")
    return nov[nov.dayofweek == 4][3]


def add_event_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    years = sorted(out[DATE_COL].dt.year.unique().tolist())
    vn_holidays = holidays.country_holidays("VN", years=years)
    out["is_vn_holiday"] = out[DATE_COL].isin(vn_holidays).astype("int8")

    mega_days = {
        "dd_3_3": (3, 3),
        "womens_day": (3, 8),
        "valentine": (2, 14),
        "dd_6_6": (6, 6),
        "dd_9_9": (9, 9),
        "dd_10_10": (10, 10),
        "dd_11_11": (11, 11),
        "dd_12_12": (12, 12),
    }
    for name, (month, day) in mega_days.items():
        out[f"is_{name}"] = ((out[DATE_COL].dt.month == month) & (out[DATE_COL].dt.day == day)).astype("int8")

    black_fridays = pd.to_datetime([_black_friday(y) for y in range(min(years), max(years) + 2)])
    out["is_black_friday"] = out[DATE_COL].isin(black_fridays).astype("int8")
    out["is_cyber_monday"] = out[DATE_COL].isin(black_fridays + pd.Timedelta(days=3)).astype("int8")

    eom = out[DATE_COL] + pd.offsets.MonthEnd(0)
    out["is_mid_month_pay"] = (out[DATE_COL].dt.day == 15).astype("int8")
    out["is_eom_pay"] = (out[DATE_COL] == eom).astype("int8")
    out["is_payday_window"] = ((out[DATE_COL].dt.day.between(14, 17)) | ((eom - out[DATE_COL]).dt.days <= 2)).astype("int8")
    return out


def build_deterministic_feature_frame(start_date: pd.Timestamp, end_date: pd.Timestamp) -> pd.DataFrame:
    base = pd.DataFrame({DATE_COL: pd.date_range(start_date, end_date, freq="D")})
    base = add_basic_calendar(base)
    base = add_cyclical(base)
    base = add_tet_features(base)
    base = add_event_features(base)
    return base.set_index(DATE_COL)


def _safe_table(df: Optional[pd.DataFrame], date_col: str) -> Optional[pd.DataFrame]:
    if df is None or df.empty:
        return None
    out = df.copy()
    out.columns = [c.strip().lower() for c in out.columns]
    if date_col not in out.columns:
        return None
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce")
    out = out.dropna(subset=[date_col])
    return out


def add_web_features(det: pd.DataFrame, web: Optional[pd.DataFrame]) -> pd.DataFrame:
    web = _safe_table(web, "date")
    if web is None:
        return det
    agg_ops: Dict[str, str] = {}
    for col, fn in [
        ("sessions", "sum"),
        ("unique_visitors", "sum"),
        ("page_views", "sum"),
        ("bounce_rate", "mean"),
        ("avg_session_duration_sec", "mean"),
    ]:
        if col in web.columns:
            agg_ops[col] = fn
    if not agg_ops:
        return det
    daily = web.groupby("date", as_index=False).agg(agg_ops)
    feat = det.reset_index().merge(daily, on="date", how="left")
    base_cols = [c for c in agg_ops]
    for col in base_cols:
        for lag in [1, 2, 3, 7, 14, 28]:
            feat[f"{col}_lag_{lag}"] = feat[col].shift(lag)
        feat[f"{col}_rmean_7"] = feat[col].shift(1).rolling(7, min_periods=3).mean()
        feat[f"{col}_rmean_28"] = feat[col].shift(1).rolling(28, min_periods=3).mean()
        feat[f"{col}_yoy"] = feat[col].shift(365)
    feat = feat.drop(columns=base_cols)
    return feat.set_index("date")


def _expand_promotions(promos: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for r in promos.itertuples(index=False):
        s = getattr(r, "start_date", pd.NaT)
        e = getattr(r, "end_date", pd.NaT)
        if pd.isna(s) or pd.isna(e):
            continue
        if e < s:
            s, e = e, s
        for dt in pd.date_range(s, e, freq="D"):
            rows.append(
                {
                    "date": dt,
                    "promo_id": getattr(r, "promo_id", None),
                    "discount_value": float(getattr(r, "discount_value", 0.0) or 0.0),
                    "promo_channel": str(getattr(r, "promo_channel", "unknown")),
                    "category": str(getattr(r, "applicable_category", "unknown")),
                    "stackable": int(getattr(r, "stackable_flag", 0) or 0),
                }
            )
    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=["date"])


def add_promo_features(det: pd.DataFrame, promos: Optional[pd.DataFrame]) -> pd.DataFrame:
    promos = _safe_table(promos, "start_date")
    if promos is None:
        return det
    if "end_date" not in promos.columns:
        promos["end_date"] = promos["start_date"]
    else:
        promos["end_date"] = pd.to_datetime(promos["end_date"], errors="coerce")
    dp = _expand_promotions(promos)
    if dp.empty:
        return det
    agg = dp.groupby("date", as_index=False).agg(
        n_active_promos=("promo_id", "nunique"),
        max_discount=("discount_value", "max"),
        mean_discount=("discount_value", "mean"),
        n_channels=("promo_channel", "nunique"),
        n_categories=("category", "nunique"),
        n_stackable=("stackable", "sum"),
    )
    feat = det.reset_index().merge(agg, on="date", how="left")
    fill_cols = ["n_active_promos", "max_discount", "mean_discount", "n_channels", "n_categories", "n_stackable"]
    feat[fill_cols] = feat[fill_cols].fillna(0)
    feat["is_active_promo"] = (feat["n_active_promos"] > 0).astype("int8")
    for w in [7, 14, 30]:
        feat[f"promo_days_last_{w}"] = feat["is_active_promo"].shift(1).rolling(w, min_periods=1).sum()
    for base in ["is_tet_week", "is_dd_11_11", "is_dd_12_12", "is_weekend"]:
        if base in feat.columns:
            feat[f"promo_x_{base}"] = feat["is_active_promo"] * feat[base]
    return feat.set_index("date")


def add_orders_features(det: pd.DataFrame, orders: Optional[pd.DataFrame]) -> pd.DataFrame:
    orders = _safe_table(orders, "order_date")
    if orders is None:
        return det
    daily = orders.groupby("order_date", as_index=False).agg(
        n_orders=("order_id", "count") if "order_id" in orders.columns else ("order_date", "count"),
        n_customers=("customer_id", pd.Series.nunique) if "customer_id" in orders.columns else ("order_date", "count"),
    )
    daily = daily.rename(columns={"order_date": "date"})
    feat = det.reset_index().merge(daily, on="date", how="left")
    for c in ["n_orders", "n_customers"]:
        feat[c] = feat[c].fillna(0.0)
        feat[f"{c}_lag_1"] = feat[c].shift(1)
        feat[f"{c}_lag_7"] = feat[c].shift(7)
        feat[f"{c}_rmean_28"] = feat[c].shift(1).rolling(28, min_periods=3).mean()
    feat = feat.drop(columns=["n_orders", "n_customers"])
    return feat.set_index("date")


def add_inventory_features(det: pd.DataFrame, inv: Optional[pd.DataFrame]) -> pd.DataFrame:
    inv = _safe_table(inv, "snapshot_date")
    if inv is None:
        return det
    for col in ["stock_on_hand", "units_sold", "fill_rate", "stockout_flag", "overstock_flag", "sell_through_rate"]:
        if col in inv.columns:
            inv[col] = pd.to_numeric(inv[col], errors="coerce").fillna(0.0)
    daily = inv.groupby("snapshot_date", as_index=False).agg(
        total_stock=("stock_on_hand", "sum") if "stock_on_hand" in inv.columns else ("snapshot_date", "count"),
        units_sold_inv=("units_sold", "sum") if "units_sold" in inv.columns else ("snapshot_date", "count"),
        fill_rate=("fill_rate", "mean") if "fill_rate" in inv.columns else ("snapshot_date", "count"),
        stockout_ratio=("stockout_flag", "mean") if "stockout_flag" in inv.columns else ("snapshot_date", "count"),
        sell_through=("sell_through_rate", "mean") if "sell_through_rate" in inv.columns else ("snapshot_date", "count"),
    )
    daily = daily.rename(columns={"snapshot_date": "date"})
    feat = det.reset_index().merge(daily, on="date", how="left")
    keep = ["total_stock", "units_sold_inv", "fill_rate", "stockout_ratio", "sell_through"]
    feat[keep] = feat[keep].ffill()
    for col in keep:
        feat[f"{col}_lag1"] = feat[col].shift(1)
        feat[f"{col}_rmean_28"] = feat[col].shift(1).rolling(28, min_periods=3).mean()
    feat = feat.drop(columns=keep)
    return feat.set_index("date")


def add_order_items_features(
    det: pd.DataFrame,
    order_items: Optional[pd.DataFrame],
    orders: Optional[pd.DataFrame],
    products: Optional[pd.DataFrame],
) -> pd.DataFrame:
    if order_items is None or orders is None:
        return det
    oi = order_items.copy()
    oi.columns = [c.strip().lower() for c in oi.columns]
    ords = _safe_table(orders, "order_date")
    if ords is None or "order_id" not in ords.columns or "order_id" not in oi.columns:
        return det

    merged = oi.merge(ords[["order_id", "order_date"]], on="order_id", how="left")
    merged = merged.dropna(subset=["order_date"])
    if products is not None:
        prod = products.copy()
        prod.columns = [c.strip().lower() for c in prod.columns]
        if "product_id" in merged.columns and "product_id" in prod.columns and "cogs" in prod.columns:
            merged = merged.merge(prod[["product_id", "cogs"]], on="product_id", how="left")
    for col in ["quantity", "unit_price", "discount_amount", "cogs"]:
        if col in merged.columns:
            merged[col] = pd.to_numeric(merged[col], errors="coerce").fillna(0.0)
        else:
            merged[col] = 0.0
    merged["line_revenue"] = (merged["quantity"] * merged["unit_price"] - merged["discount_amount"]).clip(lower=0.0)
    merged["line_cogs"] = (merged["quantity"] * merged["cogs"]).clip(lower=0.0)
    daily = merged.groupby("order_date", as_index=False).agg(
        units_sold=("quantity", "sum"),
        gross_sales_items=("line_revenue", "sum"),
        gross_cogs_items=("line_cogs", "sum"),
        discount_items=("discount_amount", "sum"),
    )
    daily = daily.rename(columns={"order_date": "date"})
    feat = det.reset_index().merge(daily, on="date", how="left")
    base_cols = ["units_sold", "gross_sales_items", "gross_cogs_items", "discount_items"]
    feat[base_cols] = feat[base_cols].fillna(0.0)
    for col in base_cols:
        feat[f"{col}_lag_1"] = feat[col].shift(1)
        feat[f"{col}_lag_7"] = feat[col].shift(7)
        feat[f"{col}_rmean_28"] = feat[col].shift(1).rolling(28, min_periods=3).mean()
    feat = feat.drop(columns=base_cols)
    return feat.set_index("date")


def add_returns_features(det: pd.DataFrame, returns: Optional[pd.DataFrame]) -> pd.DataFrame:
    returns = _safe_table(returns, "return_date")
    if returns is None:
        return det
    if "refund_amount" in returns.columns:
        returns["refund_amount"] = pd.to_numeric(returns["refund_amount"], errors="coerce").fillna(0.0)
    if "return_quantity" in returns.columns:
        returns["return_quantity"] = pd.to_numeric(returns["return_quantity"], errors="coerce").fillna(0.0)
    daily = returns.groupby("return_date", as_index=False).agg(
        n_returns=("return_id", "count") if "return_id" in returns.columns else ("order_id", "count"),
        return_qty=("return_quantity", "sum") if "return_quantity" in returns.columns else ("order_id", "count"),
        refund_amount=("refund_amount", "sum") if "refund_amount" in returns.columns else ("order_id", "count"),
    )
    daily = daily.rename(columns={"return_date": "date"})
    feat = det.reset_index().merge(daily, on="date", how="left")
    for c in ["n_returns", "return_qty", "refund_amount"]:
        feat[c] = feat[c].fillna(0.0)
        feat[f"{c}_lag_1"] = feat[c].shift(1)
        feat[f"{c}_rmean_28"] = feat[c].shift(1).rolling(28, min_periods=3).mean()
    feat = feat.drop(columns=["n_returns", "return_qty", "refund_amount"])
    return feat.set_index("date")


def add_reviews_features(det: pd.DataFrame, reviews: Optional[pd.DataFrame]) -> pd.DataFrame:
    reviews = _safe_table(reviews, "review_date")
    if reviews is None:
        return det
    if "rating" in reviews.columns:
        reviews["rating"] = pd.to_numeric(reviews["rating"], errors="coerce")
    daily = reviews.groupby("review_date", as_index=False).agg(
        n_reviews=("review_id", "count") if "review_id" in reviews.columns else ("order_id", "count"),
        rating_mean=("rating", "mean") if "rating" in reviews.columns else ("order_id", "count"),
    )
    daily = daily.rename(columns={"review_date": "date"})
    feat = det.reset_index().merge(daily, on="date", how="left")
    feat[["n_reviews", "rating_mean"]] = feat[["n_reviews", "rating_mean"]].fillna(0.0)
    for c in ["n_reviews", "rating_mean"]:
        feat[f"{c}_lag_1"] = feat[c].shift(1)
        feat[f"{c}_rmean_28"] = feat[c].shift(1).rolling(28, min_periods=3).mean()
    feat = feat.drop(columns=["n_reviews", "rating_mean"])
    return feat.set_index("date")


def build_external_feature_store(det: pd.DataFrame, tables: Dict[str, Optional[pd.DataFrame]]) -> pd.DataFrame:
    out = det.copy()
    out = add_web_features(out, tables.get("web_traffic"))
    out = add_promo_features(out, tables.get("promotions"))
    out = add_orders_features(out, tables.get("orders"))
    out = add_order_items_features(out, tables.get("order_items"), tables.get("orders"), tables.get("products"))
    out = add_returns_features(out, tables.get("returns"))
    out = add_reviews_features(out, tables.get("reviews"))
    out = add_inventory_features(out, tables.get("inventory"))
    return out


def make_lag_roll_features(series: pd.Series, horizon: int = 1) -> pd.DataFrame:
    s = series.astype(float).copy()
    out = pd.DataFrame(index=series.index)
    for lag in LAGS:
        out[f"lag_{lag}"] = s.shift(lag)
    for lag in [7, 14, 21, 28, 35, 49]:
        out[f"dow_lag_{lag}"] = s.shift(lag)
    base = s.shift(horizon)
    for w in ROLL_WINDOWS:
        r = base.rolling(w, min_periods=max(3, w // 3))
        out[f"rmean_{w}"] = r.mean()
        out[f"rstd_{w}"] = r.std()
        out[f"rmin_{w}"] = r.min()
        out[f"rmax_{w}"] = r.max()
        out[f"rmed_{w}"] = r.median()
        out[f"rcv_{w}"] = out[f"rstd_{w}"] / (out[f"rmean_{w}"].abs() + 1e-6)
    for alpha in EWM_ALPHAS:
        ewm = base.ewm(alpha=alpha, adjust=False)
        out[f"ewm_a{alpha}"] = ewm.mean()
        out[f"ewm_std_a{alpha}"] = ewm.std()
    out["diff_1"] = s.shift(1) - s.shift(2)
    out["diff_7"] = s.shift(1) - s.shift(8)
    out["yoy_lag"] = s.shift(365)
    out["yoy_diff"] = s.shift(1) - s.shift(365)
    out["yoy_ratio"] = s.shift(1) / (s.shift(365) + 1.0)
    out["yoy_roll28"] = s.shift(365).rolling(28, min_periods=10).mean()
    return out


def lag_roll_row_from_history(history: pd.Series) -> Dict[str, float]:
    h = history.astype(float).dropna()
    feats: Dict[str, float] = {}
    arr = h.to_numpy()
    n = len(arr)
    for lag in LAGS:
        feats[f"lag_{lag}"] = float(arr[-lag]) if n >= lag else np.nan
    for lag in [7, 14, 21, 28, 35, 49]:
        feats[f"dow_lag_{lag}"] = float(arr[-lag]) if n >= lag else np.nan
    for w in ROLL_WINDOWS:
        min_periods = max(3, w // 3)
        if n >= min_periods:
            vals = arr[-w:]
            feats[f"rmean_{w}"] = float(np.nanmean(vals))
            feats[f"rstd_{w}"] = float(np.nanstd(vals, ddof=1)) if len(vals) > 1 else 0.0
            feats[f"rmin_{w}"] = float(np.nanmin(vals))
            feats[f"rmax_{w}"] = float(np.nanmax(vals))
            feats[f"rmed_{w}"] = float(np.nanmedian(vals))
            feats[f"rcv_{w}"] = feats[f"rstd_{w}"] / (abs(feats[f"rmean_{w}"]) + 1e-6)
        else:
            feats[f"rmean_{w}"] = np.nan
            feats[f"rstd_{w}"] = np.nan
            feats[f"rmin_{w}"] = np.nan
            feats[f"rmax_{w}"] = np.nan
            feats[f"rmed_{w}"] = np.nan
            feats[f"rcv_{w}"] = np.nan
    hs = h.copy()
    for alpha in EWM_ALPHAS:
        feats[f"ewm_a{alpha}"] = float(hs.ewm(alpha=alpha, adjust=False).mean().iloc[-1]) if n else np.nan
        feats[f"ewm_std_a{alpha}"] = float(hs.ewm(alpha=alpha, adjust=False).std().iloc[-1]) if n > 1 else 0.0
    feats["diff_1"] = float(arr[-1] - arr[-2]) if n >= 2 else np.nan
    feats["diff_7"] = float(arr[-1] - arr[-8]) if n >= 8 else np.nan
    feats["yoy_lag"] = float(arr[-365]) if n >= 365 else np.nan
    feats["yoy_diff"] = float(arr[-1] - arr[-365]) if n >= 365 else np.nan
    feats["yoy_ratio"] = float(arr[-1] / (arr[-365] + 1.0)) if n >= 365 else np.nan
    feats["yoy_roll28"] = float(np.nanmean(arr[-(365 + 28) : -365])) if n >= (365 + 10) else np.nan
    return feats


class ExpandingWindowWalkForward:
    def __init__(self, initial_train_days=365 * 5, val_days=180, step_days=180, gap_days=28, max_splits=8):
        self.initial_train_days = initial_train_days
        self.val_days = val_days
        self.step_days = step_days
        self.gap_days = gap_days
        self.max_splits = max_splits

    def split(self, X: pd.DataFrame) -> Iterator[Tuple[np.ndarray, np.ndarray]]:
        n = len(X)
        idx = np.arange(n)
        train_end = self.initial_train_days
        fold = 0
        while True:
            val_start = train_end + self.gap_days
            val_end = val_start + self.val_days
            if val_end > n or (self.max_splits and fold >= self.max_splits):
                break
            yield idx[:train_end], idx[val_start:val_end]
            train_end += self.step_days
            fold += 1


def get_xgb_device(gpu_id: int, gpu_count: int) -> str:
    if torch.cuda.is_available() and gpu_count > 0:
        return f"cuda:{gpu_id}"
    return "cpu"


def booster_predict(model: xgb.XGBRegressor, X: pd.DataFrame, gpu_id: int, gpu_count: int) -> np.ndarray:
    booster = model.get_booster()
    device = get_xgb_device(gpu_id, gpu_count)
    booster.set_param({"device": device})
    x_arr = np.asarray(X, dtype=np.float32)
    # Prefer inplace_predict to avoid repeated DMatrix allocations and device mismatch fallback.
    try:
        return booster.inplace_predict(x_arr)
    except Exception:
        dmat = xgb.DMatrix(x_arr)
        return booster.predict(dmat)


def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = np.abs(y_true) + np.abs(y_pred) + 1e-8
    return float(np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0)


def mase(y_true: np.ndarray, y_pred: np.ndarray, insample: np.ndarray, seasonal_period: int = 7) -> float:
    m = seasonal_period if len(insample) > seasonal_period + 1 else 1
    if len(insample) <= m:
        return float(np.nan)
    d = np.mean(np.abs(insample[m:] - insample[:-m]))
    if d < 1e-8:
        d = np.mean(np.abs(np.diff(insample))) + 1e-8
    return float(np.mean(np.abs(y_true - y_pred)) / d)


def evaluate_full(y_true: np.ndarray, y_pred: np.ndarray, insample: np.ndarray) -> Dict[str, float]:
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
        "sMAPE": smape(y_true, y_pred),
        "MASE": mase(y_true, y_pred, insample=insample, seasonal_period=7),
    }


def seasonal_naive_predict(history: pd.Series, forecast_dates: pd.DatetimeIndex, seasonal_lag: int = 365) -> np.ndarray:
    hist = history.copy()
    out = []
    for dt in forecast_dates:
        ref = dt - pd.Timedelta(days=seasonal_lag)
        if ref in hist.index:
            pred = float(hist.loc[ref])
        else:
            pred = float(hist.iloc[-7:].mean())
        out.append(max(pred, 0.0))
        hist.loc[dt] = pred
    return np.array(out, dtype=float)


def train_prophet_safe(train_dates: pd.Series, train_y: np.ndarray, pred_dates: pd.DatetimeIndex, seed: int) -> np.ndarray:
    try:
        m = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
            seasonality_mode="multiplicative",
            changepoint_prior_scale=0.05,
            uncertainty_samples=0,
        )
        m.add_country_holidays(country_name="VN")
        m.fit(pd.DataFrame({"ds": train_dates.values, "y": train_y}), seed=seed)
        fcst = m.predict(pd.DataFrame({"ds": pred_dates.values}))
        return np.clip(fcst["yhat"].to_numpy(dtype=float), 0.0, None)
    except Exception:
        hist = pd.Series(train_y, index=pd.DatetimeIndex(train_dates.values))
        return seasonal_naive_predict(hist, pred_dates, seasonal_lag=365)


def horizon_bucket(step: int) -> str:
    for lo, hi, name in HORIZON_BUCKETS:
        if lo <= step <= hi:
            return name
    return HORIZON_BUCKETS[-1][2]


def normalize_weights(w: np.ndarray) -> np.ndarray:
    x = np.clip(np.asarray(w, dtype=float), 0.0, None)
    s = float(x.sum())
    if s <= 1e-12:
        return np.ones_like(x, dtype=float) / len(x)
    return x / s


def combined_metric_loss(y_true: np.ndarray, y_pred: np.ndarray, alpha: float = 0.5) -> float:
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    scale_mae = float(np.mean(np.abs(y_true - np.median(y_true))) + 1e-6)
    scale_rmse = float(np.std(y_true) + 1e-6)
    return alpha * (mae / scale_mae) + (1.0 - alpha) * (rmse / scale_rmse)


def fit_convex_blend_weights(
    y_true: np.ndarray,
    pred_matrix: np.ndarray,
    alpha: float = 0.5,
    lam: float = 0.01,
    prior_strength: float = 0.02,
) -> Tuple[np.ndarray, float]:
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(pred_matrix, dtype=float)
    n_models = p.shape[1]
    prior = np.ones(n_models, dtype=float) / n_models

    def objective(raw_w: np.ndarray) -> float:
        w = normalize_weights(raw_w)
        pred = p @ w
        core = combined_metric_loss(y, pred, alpha=alpha)
        reg_l2 = lam * float(np.sum(w**2))
        reg_prior = prior_strength * float(np.sum((w - prior) ** 2))
        return core + reg_l2 + reg_prior

    init = np.ones(n_models, dtype=float) / n_models
    best_w = init.copy()
    best_loss = objective(best_w)

    if minimize is not None:
        try:
            res = minimize(
                objective,
                init,
                method="SLSQP",
                bounds=[(0.0, 1.0)] * n_models,
                constraints=[{"type": "eq", "fun": lambda w: float(np.sum(w) - 1.0)}],
                options={"maxiter": 500, "ftol": 1e-9},
            )
            if res.success:
                w = normalize_weights(res.x)
                cur = objective(w)
                if cur < best_loss:
                    best_loss = cur
                    best_w = w
        except Exception:
            pass

    # Random Dirichlet fallback / local search even when scipy is available.
    rng = np.random.default_rng(SEED)
    for _ in range(3000):
        cand = rng.dirichlet(np.ones(n_models))
        cur = objective(cand)
        if cur < best_loss:
            best_loss = cur
            best_w = cand

    return normalize_weights(best_w), float(best_loss)


def fit_bucket_blend_weights(oof: pd.DataFrame, pred_cols: List[str], lam: float = 0.01) -> Tuple[Dict[str, List[float]], np.ndarray]:
    usable = oof.dropna(subset=pred_cols + ["actual", "h_bucket"])
    y_all = usable["actual"].to_numpy(dtype=float)
    p_all = usable[pred_cols].to_numpy(dtype=float)
    global_w, _ = fit_convex_blend_weights(
        y_all,
        p_all,
        alpha=CFG.composite_alpha,
        lam=lam,
        prior_strength=CFG.dirichlet_prior_strength,
    )
    bucket_weights: Dict[str, List[float]] = {}
    for _, _, bname in HORIZON_BUCKETS:
        sub = usable[usable["h_bucket"] == bname]
        if len(sub) < 40:
            bucket_weights[bname] = global_w.tolist()
            continue
        wb, _ = fit_convex_blend_weights(
            sub["actual"].to_numpy(dtype=float),
            sub[pred_cols].to_numpy(dtype=float),
            alpha=CFG.composite_alpha,
            lam=lam,
            prior_strength=CFG.dirichlet_prior_strength,
        )
        bucket_weights[bname] = wb.tolist()
    return bucket_weights, global_w


def compute_event_multipliers(oof: pd.DataFrame, event_cols: List[str], k: float = 12.0) -> Dict[str, float]:
    out: Dict[str, float] = {}
    work = oof.copy()
    work[DATE_COL] = pd.to_datetime(work[DATE_COL], errors="coerce")
    work = work[~work[DATE_COL].dt.year.isin([2020, 2021])]
    for ev in event_cols:
        if ev not in work.columns:
            out[ev] = 1.0
            continue
        sub = work[(work[ev] == 1) & work["pred_blend"].notna()]
        if len(sub) >= 3 and sub["pred_blend"].sum() > 0:
            ratio = float(sub["actual"].sum() / sub["pred_blend"].sum())
            n = float(len(sub))
            out[ev] = float((n * ratio + k) / (n + k))
        else:
            out[ev] = 1.0
    return out


def apply_event_multiplier(pred: float, row: pd.Series, mult: Dict[str, float]) -> float:
    m = 1.0
    for ev, f in mult.items():
        if int(row.get(ev, 0)) == 1:
            m *= f
    return float(pred * m)


def fit_residual_sarima(oof: pd.DataFrame, pred_col: str = "pred_blend_event") -> Optional[SARIMAX]:
    residual = oof.dropna(subset=[pred_col]).copy()
    if residual.empty:
        return None
    residual["resid"] = residual["actual"] - residual[pred_col]
    series = residual.set_index(DATE_COL)["resid"].asfreq("D").fillna(0.0)
    if len(series) < 365:
        return None
    lb = acorr_ljungbox(series, lags=[7, 14, 28], return_df=True)
    if not (lb["lb_pvalue"] < 0.05).any():
        return None
    try:
        return SARIMAX(
            series,
            order=(1, 0, 1),
            seasonal_order=(1, 0, 1, 7),
            enforce_stationarity=False,
            enforce_invertibility=False,
        ).fit(disp=False)
    except Exception:
        return None


def export_shap_importance(model: xgb.XGBRegressor, X: pd.DataFrame, out_path: Path) -> None:
    try:
        import shap

        sample = X.sample(min(3000, len(X)), random_state=SEED)
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(sample, check_additivity=False)
        mean_abs = np.abs(shap_values).mean(axis=0)
        shap_df = pd.DataFrame({"feature": sample.columns, "mean_abs_shap": mean_abs}).sort_values("mean_abs_shap", ascending=False)
        shap_df.head(120).to_csv(out_path, index=False)
    except Exception:
        pass


def duan_smearing_factor(y_true: np.ndarray, pred_log1p: np.ndarray) -> float:
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(pred_log1p, dtype=float)
    resid = np.log1p(np.clip(y, 0.0, None)) - p
    smear = float(np.mean(np.exp(np.clip(resid, -6.0, 6.0))))
    return float(np.clip(smear, 0.7, 3.0))


def predict_from_log1p_with_smear(pred_log1p: np.ndarray, smear: float) -> np.ndarray:
    # E[y+1|x] ~= exp(E[log(1+y)|x]) * E[exp(residual)].
    out = np.exp(np.asarray(pred_log1p, dtype=float)) * float(smear) - 1.0
    return np.clip(out, 0.0, None)


def fit_linear_calibration(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[float, float]:
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(y_pred, dtype=float)
    if len(y) < 30:
        return 1.0, 0.0
    try:
        reg = LinearRegression().fit(p.reshape(-1, 1), y)
        a = float(reg.coef_[0])
        b = float(reg.intercept_)
        if not np.isfinite(a):
            a = 1.0
        if not np.isfinite(b):
            b = 0.0
        return a, b
    except Exception:
        return 1.0, 0.0


def apply_linear_calibration(y_pred: np.ndarray, a: float, b: float) -> np.ndarray:
    return np.asarray(y_pred, dtype=float) * float(a) + float(b)


def winsor_bounds(y_train: np.ndarray) -> Tuple[float, float]:
    lo_q = float(np.quantile(y_train, CFG.winsor_q_low))
    hi_q = float(np.quantile(y_train, CFG.winsor_q_high))
    lo = max(lo_q * float(CFG.lower_clip_scale), 0.0)
    hi = max(hi_q * float(CFG.upper_clip_scale), lo + 1.0)
    return lo, hi


def clip_with_growth(pred: float, lo: float, hi: float, h_step: int) -> float:
    growth = float((1.0 + CFG.growth_headroom_annual) ** (float(h_step) / 365.0))
    return float(np.clip(pred, lo, hi * growth))


def _stacker_feature_frame(
    pred_base: np.ndarray,
    h_step: np.ndarray,
    dates: pd.Series,
    oof_frame: pd.DataFrame,
    event_cols: List[str],
) -> pd.DataFrame:
    dt = pd.to_datetime(dates, errors="coerce")
    feat = pd.DataFrame(
        {
            "pred_base": np.asarray(pred_base, dtype=float),
            "h_step": np.asarray(h_step, dtype=float),
            "month": dt.dt.month.astype(float).fillna(0.0),
            "dow": dt.dt.dayofweek.astype(float).fillna(0.0),
        },
        index=oof_frame.index if hasattr(oof_frame, "index") else None,
    )
    for c in event_cols:
        if c in oof_frame.columns:
            feat[c] = oof_frame[c].astype(float).fillna(0.0).to_numpy()
        else:
            feat[c] = 0.0
    return feat


def fit_residual_stacker(
    oof: pd.DataFrame,
    event_cols: List[str],
    gpu_id: int,
    gpu_count: int,
    seed: int,
) -> Tuple[Optional[xgb.XGBRegressor], List[str], Tuple[float, float]]:
    usable = oof.dropna(subset=["actual", "pred_final_base", "h_step", DATE_COL]).copy()
    if len(usable) < 200:
        return None, [], (-np.inf, np.inf)

    X = _stacker_feature_frame(
        pred_base=usable["pred_final_base"].to_numpy(dtype=float),
        h_step=usable["h_step"].to_numpy(dtype=float),
        dates=usable[DATE_COL],
        oof_frame=usable,
        event_cols=event_cols,
    )
    y_resid = (usable["actual"] - usable["pred_final_base"]).to_numpy(dtype=float)

    n = len(usable)
    cut = max(int(n * 0.8), n - 400)
    cut = min(max(cut, 100), n - 50)
    X_tr, X_va = X.iloc[:cut], X.iloc[cut:]
    y_tr, y_va = y_resid[:cut], y_resid[cut:]

    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        n_estimators=1200,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=15,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=5.0,
        reg_alpha=0.05,
        tree_method="hist",
        device=get_xgb_device(gpu_id, gpu_count),
        n_jobs=CFG.xgb_n_jobs,
        random_state=seed,
        early_stopping_rounds=120,
    )
    try:
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    except Exception:
        try:
            model.fit(X, y_resid, verbose=False)
        except Exception:
            return None, [], (-np.inf, np.inf)

    lo = float(np.quantile(y_resid, 0.01))
    hi = float(np.quantile(y_resid, 0.99))
    return model, X.columns.tolist(), (lo, hi)


def tune_xgb_params(
    train_df: pd.DataFrame,
    target_col: str,
    feature_cols: List[str],
    splitter: ExpandingWindowWalkForward,
    gpu_id: int,
    gpu_count: int,
    seed: int,
) -> Dict[str, float]:
    default_params = {
        "learning_rate": 0.03,
        "max_depth": 8,
        "min_child_weight": 20,
        "subsample": 0.8,
        "colsample_bytree": 0.9,
        "reg_alpha": 0.01,
        "reg_lambda": 0.1,
        "gamma": 0.0,
        "tweedie_variance_power": 1.3,
        "huber_slope": float(CFG.huber_slope),
    }
    if (not CFG.enable_optuna) or CFG.n_trials <= 0:
        return default_params
    try:
        import optuna
    except Exception:
        return default_params

    y_full = train_df[target_col].to_numpy(dtype=float)
    folds = list(splitter.split(train_df))
    if not folds:
        return default_params

    def objective(trial: "optuna.Trial") -> float:
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
            "max_depth": trial.suggest_int("max_depth", 4, 12),
            "min_child_weight": trial.suggest_int("min_child_weight", 5, 40),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 2.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
            "gamma": trial.suggest_float("gamma", 0.0, 3.0),
            "tweedie_variance_power": trial.suggest_float("tweedie_variance_power", 1.1, 1.3),
            "huber_slope": trial.suggest_float("huber_slope", 2e5, 1.2e6, log=True),
        }
        fold_losses = []
        for fold_id, (tr_idx, va_idx) in enumerate(folds):
            X_tr = train_df.loc[tr_idx, feature_cols]
            X_va = train_df.loc[va_idx, feature_cols]
            y_tr = y_full[tr_idx]
            y_va = y_full[va_idx]
            model = xgb.XGBRegressor(
                n_estimators=min(1500, CFG.xgb_cv_estimators),
                learning_rate=params["learning_rate"],
                max_depth=int(params["max_depth"]),
                min_child_weight=int(params["min_child_weight"]),
                subsample=params["subsample"],
                colsample_bytree=params["colsample_bytree"],
                reg_alpha=params["reg_alpha"],
                reg_lambda=params["reg_lambda"],
                gamma=params["gamma"],
                objective="reg:squarederror",
                eval_metric="mae",
                tree_method="hist",
                device=get_xgb_device(gpu_id, gpu_count),
                n_jobs=CFG.xgb_n_jobs,
                random_state=seed + fold_id,
                early_stopping_rounds=100,
            )
            model.fit(X_tr, np.log1p(np.clip(y_tr, 0, None)), eval_set=[(X_va, np.log1p(np.clip(y_va, 0, None)))], verbose=False)
            pred_log = booster_predict(model, X_va, gpu_id, gpu_count)
            if CFG.use_duan_smearing:
                pred_log_tr = booster_predict(model, X_tr, gpu_id, gpu_count)
                smear = duan_smearing_factor(y_tr, pred_log_tr)
            else:
                smear = 1.0
            pred = predict_from_log1p_with_smear(pred_log, smear)
            fold_loss = combined_metric_loss(y_va, np.clip(pred, 0.0, None), alpha=CFG.composite_alpha)
            fold_losses.append(float(fold_loss))
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            trial.report(float(np.mean(fold_losses)), step=fold_id)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(fold_losses))

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=seed, multivariate=True, n_startup_trials=10),
        pruner=optuna.pruners.HyperbandPruner(min_resource=1, max_resource=max(1, len(folds))),
    )
    study.optimize(
        objective,
        n_trials=CFG.n_trials,
        timeout=(None if CFG.optuna_timeout <= 0 else CFG.optuna_timeout),
        show_progress_bar=False,
        gc_after_trial=True,
    )
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    best = default_params.copy()
    best.update(study.best_params)
    return best


def build_xgb_model(
    params: Dict[str, float],
    objective: str,
    eval_metric: str,
    seed: int,
    gpu_id: int,
    gpu_count: int,
    n_estimators: int,
    early_stopping_rounds: Optional[int],
) -> xgb.XGBRegressor:
    cfg = {
        "n_estimators": int(n_estimators),
        "learning_rate": float(params["learning_rate"]),
        "max_depth": int(params["max_depth"]),
        "min_child_weight": int(params["min_child_weight"]),
        "subsample": float(params["subsample"]),
        "colsample_bytree": float(params["colsample_bytree"]),
        "reg_alpha": float(params["reg_alpha"]),
        "reg_lambda": float(params["reg_lambda"]),
        "gamma": float(params["gamma"]),
        "objective": objective,
        "eval_metric": eval_metric,
        "tree_method": "hist",
        "device": get_xgb_device(gpu_id, gpu_count),
        "n_jobs": CFG.xgb_n_jobs,
        "random_state": seed,
    }
    if objective == "reg:tweedie":
        cfg["tweedie_variance_power"] = float(params.get("tweedie_variance_power", 1.3))
    if objective == "reg:pseudohubererror":
        cfg["huber_slope"] = float(params.get("huber_slope", CFG.huber_slope))
    if early_stopping_rounds is not None:
        cfg["early_stopping_rounds"] = int(early_stopping_rounds)
    return xgb.XGBRegressor(**cfg)


def train_target(
    target_col: str,
    gpu_id: int,
    gpu_count: int,
    seed: int,
    sales_df: pd.DataFrame,
    deterministic_features: pd.DataFrame,
    forecast_dates: pd.DatetimeIndex,
    output_dir: Path,
) -> Dict[str, object]:
    frame = sales_df[[DATE_COL, target_col]].copy().sort_values(DATE_COL).reset_index(drop=True)
    target_series = pd.Series(frame[target_col].to_numpy(dtype=float), index=pd.DatetimeIndex(frame[DATE_COL]), name=target_col)
    if target_col == "cogs_ratio":
        target_series = target_series.clip(0.01, 3.0)

    lag_feats = make_lag_roll_features(target_series, horizon=1)
    train_df = deterministic_features.loc[target_series.index].copy().join(lag_feats)
    train_df[DATE_COL] = train_df.index
    train_df[target_col] = target_series.values
    min_needed = [c for c in ["lag_365", "rmean_90", "yoy_lag", "yoy_roll28"] if c in train_df.columns]
    if min_needed:
        train_df = train_df.dropna(subset=min_needed)
    train_df = train_df.reset_index(drop=True).sort_values(DATE_COL).reset_index(drop=True)

    if CFG.train_recent_years is not None and CFG.train_recent_years > 0:
        warm_days = int(max(LAGS) + max(ROLL_WINDOWS) + 30)
        cutoff = pd.to_datetime(train_df[DATE_COL].max()) - pd.Timedelta(days=int(365 * CFG.train_recent_years + warm_days))
        reduced = train_df[train_df[DATE_COL] >= cutoff].copy()
        min_rows = int(CFG.initial_train_days + CFG.gap_days + CFG.val_days + 30)
        if len(reduced) >= min_rows:
            train_df = reduced.reset_index(drop=True)

    feature_cols = [c for c in train_df.columns if c not in [DATE_COL, target_col]]
    train_df = train_df.replace([np.inf, -np.inf], np.nan)
    med = train_df[feature_cols].median(numeric_only=True)
    train_df[feature_cols] = train_df[feature_cols].fillna(med).astype(np.float32)
    train_df[target_col] = train_df[target_col].astype(np.float32)

    splitter = ExpandingWindowWalkForward(
        initial_train_days=CFG.initial_train_days,
        val_days=CFG.val_days,
        step_days=CFG.step_days,
        gap_days=CFG.gap_days,
        max_splits=CFG.cv_max_splits,
    )
    folds = list(splitter.split(train_df))
    if not folds:
        raise RuntimeError(f"Not enough rows for CV: {target_col}")

    best_params = tune_xgb_params(train_df, target_col, feature_cols, splitter, gpu_id, gpu_count, seed)
    ev_cols = [c for c in EVENT_COLS if c in train_df.columns]

    oof = train_df[[DATE_COL, target_col] + ev_cols].copy().rename(columns={target_col: "actual"})
    oof["pred_l1"] = np.nan
    oof["pred_tweedie"] = np.nan
    oof["pred_prophet"] = np.nan
    if CFG.use_huber_model:
        oof["pred_huber"] = np.nan
    oof["h_step"] = np.nan
    oof["h_bucket"] = None

    fold_smears: List[float] = []
    for fold_id, (tr_idx, va_idx) in enumerate(folds):
        X_tr = train_df.loc[tr_idx, feature_cols]
        X_va = train_df.loc[va_idx, feature_cols]
        y_tr = train_df.loc[tr_idx, target_col].to_numpy(dtype=np.float32)
        y_va = train_df.loc[va_idx, target_col].to_numpy(dtype=np.float32)

        h_steps = np.arange(1, len(va_idx) + 1)
        oof.loc[va_idx, "h_step"] = h_steps
        oof.loc[va_idx, "h_bucket"] = [horizon_bucket(int(h)) for h in h_steps]

        l1 = build_xgb_model(best_params, "reg:squarederror", "mae", seed + fold_id, gpu_id, gpu_count, CFG.xgb_cv_estimators, 120)
        l1.fit(X_tr, np.log1p(np.clip(y_tr, 0, None)), eval_set=[(X_va, np.log1p(np.clip(y_va, 0, None)))], verbose=False)
        pred_l1_log = booster_predict(l1, X_va, gpu_id, gpu_count)
        if CFG.use_duan_smearing:
            pred_l1_log_tr = booster_predict(l1, X_tr, gpu_id, gpu_count)
            smear = duan_smearing_factor(y_tr, pred_l1_log_tr)
        else:
            smear = 1.0
        fold_smears.append(smear)
        pred_l1 = predict_from_log1p_with_smear(pred_l1_log, smear)
        oof.loc[va_idx, "pred_l1"] = pred_l1

        tw = build_xgb_model(best_params, "reg:tweedie", "rmse", seed + 101 + fold_id, gpu_id, gpu_count, CFG.xgb_cv_estimators, 120)
        tw.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        oof.loc[va_idx, "pred_tweedie"] = np.clip(booster_predict(tw, X_va, gpu_id, gpu_count), 0.0, None)

        if CFG.use_huber_model:
            hb = build_xgb_model(best_params, "reg:pseudohubererror", "rmse", seed + 201 + fold_id, gpu_id, gpu_count, CFG.xgb_cv_estimators, 120)
            hb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            oof.loc[va_idx, "pred_huber"] = np.clip(booster_predict(hb, X_va, gpu_id, gpu_count), 0.0, None)
            del hb

        if CFG.use_prophet_cv:
            pred_prophet = train_prophet_safe(train_df.loc[tr_idx, DATE_COL], y_tr, pd.DatetimeIndex(train_df.loc[va_idx, DATE_COL].values), seed)
        else:
            hist_cv = pd.Series(y_tr, index=pd.DatetimeIndex(train_df.loc[tr_idx, DATE_COL].values))
            pred_prophet = seasonal_naive_predict(hist_cv, pd.DatetimeIndex(train_df.loc[va_idx, DATE_COL].values), seasonal_lag=365)
        if target_col == "cogs_ratio":
            pred_prophet = np.clip(pred_prophet, 0.01, 3.0)
        oof.loc[va_idx, "pred_prophet"] = pred_prophet

        del l1, tw
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pred_cols = ["pred_l1", "pred_tweedie"]
    if CFG.use_huber_model:
        pred_cols.append("pred_huber")
    pred_cols.append("pred_prophet")

    bucket_weights, global_w = fit_bucket_blend_weights(oof.dropna(subset=pred_cols + ["actual", "h_bucket"]), pred_cols, lam=CFG.lam_blend)

    def blend_row(row: pd.Series) -> float:
        w = np.array(bucket_weights.get(row["h_bucket"], global_w.tolist()), dtype=float)
        p = np.array([float(row[c]) for c in pred_cols], dtype=float)
        return float(np.dot(w, p))

    oof["pred_blend"] = oof.apply(lambda r: blend_row(r) if pd.notna(r[pred_cols[0]]) else np.nan, axis=1)
    event_mult = compute_event_multipliers(oof, ev_cols)
    oof["pred_blend_event"] = oof.apply(
        lambda r: apply_event_multiplier(float(r["pred_blend"]), r, event_mult) if pd.notna(r["pred_blend"]) else np.nan,
        axis=1,
    )

    # Post-hoc calibration on OOF (high ROI per optimize_2 plan).
    cal_a, cal_b = 1.0, 0.0
    usable_cal = oof.dropna(subset=["actual", "pred_blend_event"])
    if CFG.use_linear_calibration and len(usable_cal) >= 50:
        cal_a, cal_b = fit_linear_calibration(usable_cal["actual"].to_numpy(dtype=float), usable_cal["pred_blend_event"].to_numpy(dtype=float))
    oof["pred_calibrated"] = np.nan
    mask_pred = oof["pred_blend_event"].notna()
    oof.loc[mask_pred, "pred_calibrated"] = apply_linear_calibration(oof.loc[mask_pred, "pred_blend_event"].to_numpy(dtype=float), cal_a, cal_b)

    y_all_np = train_df[target_col].to_numpy(dtype=float)
    clip_lo, clip_hi = winsor_bounds(y_all_np)
    oof["pred_final_base"] = oof["pred_calibrated"]
    if CFG.use_prediction_winsor:
        oof.loc[mask_pred, "pred_final_base"] = np.clip(oof.loc[mask_pred, "pred_final_base"], clip_lo, clip_hi)
    oof["pred_final_base"] = np.clip(oof["pred_final_base"], 0.0, None)
    if target_col == "cogs_ratio":
        oof["pred_final_base"] = np.clip(oof["pred_final_base"], 0.01, 3.0)

    residual_model = fit_residual_sarima(oof, pred_col="pred_final_base") if CFG.use_residual_sarima else None
    stacker_model = None
    stacker_cols: List[str] = []
    stacker_clip = (-np.inf, np.inf)
    if CFG.use_residual_stacker:
        stacker_model, stacker_cols, stacker_clip = fit_residual_stacker(oof, ev_cols, gpu_id, gpu_count, seed)

    X_all = train_df[feature_cols].astype(np.float32)
    y_all = train_df[target_col].to_numpy(dtype=np.float32)
    seeds = CFG.bagging_seeds if CFG.use_multi_seed else [seed]

    l1_models: List[xgb.XGBRegressor] = []
    l1_smears: List[float] = []
    tw_models: List[xgb.XGBRegressor] = []
    hb_models: List[xgb.XGBRegressor] = []

    for s in seeds:
        l1m = build_xgb_model(best_params, "reg:squarederror", "mae", s, gpu_id, gpu_count, CFG.xgb_final_estimators, None)
        l1m.fit(X_all, np.log1p(np.clip(y_all, 0, None)), verbose=False)
        l1_models.append(l1m)
        if CFG.use_duan_smearing:
            pred_tr_log = booster_predict(l1m, X_all, gpu_id, gpu_count)
            l1_smears.append(duan_smearing_factor(y_all, pred_tr_log))
        else:
            l1_smears.append(1.0)

        twm = build_xgb_model(best_params, "reg:tweedie", "rmse", s + 11, gpu_id, gpu_count, CFG.xgb_final_estimators, None)
        twm.fit(X_all, y_all, verbose=False)
        tw_models.append(twm)

        if CFG.use_huber_model:
            hbm = build_xgb_model(best_params, "reg:pseudohubererror", "rmse", s + 21, gpu_id, gpu_count, CFG.xgb_final_estimators, None)
            hbm.fit(X_all, y_all, verbose=False)
            hb_models.append(hbm)

    if CFG.use_prophet_final:
        prophet_arr = train_prophet_safe(train_df[DATE_COL], y_all, forecast_dates, seed=seed)
    else:
        hist_full = pd.Series(y_all, index=pd.DatetimeIndex(train_df[DATE_COL].values))
        prophet_arr = seasonal_naive_predict(hist_full, forecast_dates, seasonal_lag=365)
    if target_col == "cogs_ratio":
        prophet_arr = np.clip(prophet_arr, 0.01, 3.0)
    prophet_full = pd.Series(prophet_arr, index=forecast_dates)

    residual_fcst = None
    if residual_model is not None:
        try:
            residual_fcst = pd.Series(
                residual_model.get_forecast(steps=len(forecast_dates)).predicted_mean.to_numpy(dtype=float),
                index=forecast_dates,
            )
        except Exception:
            residual_fcst = None

    history = target_series.copy()
    preds = []
    for h_step, dt in enumerate(forecast_dates, start=1):
        row = deterministic_features.loc[dt].to_dict()
        row.update(lag_roll_row_from_history(history.iloc[-FORECAST_HISTORY_WINDOW:]))
        X_row = pd.DataFrame([{c: row.get(c, np.nan) for c in feature_cols}]).fillna(med).astype(np.float32)

        p1 = float(np.mean([predict_from_log1p_with_smear(booster_predict(m, X_row, gpu_id, gpu_count), s)[0] for m, s in zip(l1_models, l1_smears)]))
        p2 = float(np.mean([booster_predict(m, X_row, gpu_id, gpu_count)[0] for m in tw_models]))
        p_prophet = float(prophet_full.loc[dt]) if dt in prophet_full.index else p2

        pred_map: Dict[str, float] = {
            "pred_l1": max(p1, 0.0),
            "pred_tweedie": max(p2, 0.0),
            "pred_prophet": max(p_prophet, 0.0),
        }
        if CFG.use_huber_model:
            p_hb = float(np.mean([booster_predict(m, X_row, gpu_id, gpu_count)[0] for m in hb_models])) if hb_models else p2
            pred_map["pred_huber"] = max(p_hb, 0.0)

        w = np.array(bucket_weights.get(horizon_bucket(int(h_step)), global_w.tolist()), dtype=float)
        pred_vec = np.array([pred_map[c] for c in pred_cols], dtype=float)
        pred = float(np.dot(w, pred_vec))
        pred = apply_event_multiplier(pred, deterministic_features.loc[dt], event_mult)
        pred = float(apply_linear_calibration(np.array([pred]), cal_a, cal_b)[0])
        if CFG.use_prediction_winsor:
            pred = clip_with_growth(pred, clip_lo, clip_hi, h_step)

        if residual_fcst is not None and dt in residual_fcst.index:
            pred += float(residual_fcst.loc[dt])

        if stacker_model is not None and stacker_cols:
            stack_row = {
                "pred_base": pred,
                "h_step": float(h_step),
                "month": float(pd.Timestamp(dt).month),
                "dow": float(pd.Timestamp(dt).dayofweek),
            }
            for c in ev_cols:
                stack_row[c] = float(deterministic_features.loc[dt].get(c, 0.0))
            Xs = pd.DataFrame([{c: stack_row.get(c, 0.0) for c in stacker_cols}])
            corr = float(stacker_model.predict(Xs)[0])
            corr = float(np.clip(corr, stacker_clip[0], stacker_clip[1]))
            pred += corr

        pred = max(pred, 0.0)
        if target_col == "cogs_ratio":
            pred = float(np.clip(pred, 0.01, 3.0))
        preds.append(pred)
        history.loc[dt] = pred

    forecast = pd.Series(preds, index=forecast_dates)
    eval_required = pred_cols + ["pred_blend", "pred_blend_event", "pred_final_base", "actual"]
    eval_rows = oof.dropna(subset=eval_required).copy()
    insample = target_series.to_numpy(dtype=float)
    metrics = {
        "l1": evaluate_full(eval_rows["actual"].to_numpy(), eval_rows["pred_l1"].to_numpy(), insample),
        "tweedie": evaluate_full(eval_rows["actual"].to_numpy(), eval_rows["pred_tweedie"].to_numpy(), insample),
        "prophet_or_sn": evaluate_full(eval_rows["actual"].to_numpy(), eval_rows["pred_prophet"].to_numpy(), insample),
        "blend": evaluate_full(eval_rows["actual"].to_numpy(), eval_rows["pred_blend"].to_numpy(), insample),
        "blend_event": evaluate_full(eval_rows["actual"].to_numpy(), eval_rows["pred_blend_event"].to_numpy(), insample),
        "final_calibrated": evaluate_full(eval_rows["actual"].to_numpy(), eval_rows["pred_final_base"].to_numpy(), insample),
    }
    if CFG.use_huber_model and "pred_huber" in eval_rows.columns:
        metrics["huber"] = evaluate_full(eval_rows["actual"].to_numpy(), eval_rows["pred_huber"].to_numpy(), insample)

    bucket_metrics = {}
    for _, _, bname in HORIZON_BUCKETS:
        sub = eval_rows[eval_rows["h_bucket"] == bname]
        if len(sub) < 20:
            continue
        bucket_metrics[bname] = evaluate_full(sub["actual"].to_numpy(), sub["pred_final_base"].to_numpy(), insample)
    metrics["bucket_final"] = bucket_metrics

    oof.to_csv(output_dir / f"oof_{target_col}.csv", index=False)
    fi = pd.DataFrame(
        {
            "feature": feature_cols,
            "importance_gain_l1": l1_models[0].feature_importances_,
            "importance_gain_tweedie": tw_models[0].feature_importances_,
        }
    )
    if CFG.use_huber_model and hb_models:
        fi["importance_gain_huber"] = hb_models[0].feature_importances_
    fi = fi.sort_values("importance_gain_l1", ascending=False)
    fi.to_csv(output_dir / f"feature_importance_{target_col}.csv", index=False)
    export_shap_importance(l1_models[0], X_all, output_dir / f"shap_{target_col}.csv")

    del l1_models, tw_models, hb_models
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "target": target_col,
        "gpu_id": gpu_id,
        "forecast": forecast,
        "oof_frame": oof,
        "pred_cols": pred_cols,
        "blend_weights_global": {pred_cols[i]: float(global_w[i]) for i in range(len(pred_cols))},
        "blend_weights_bucket": bucket_weights,
        "event_multipliers": event_mult,
        "calibration": {"a": float(cal_a), "b": float(cal_b)},
        "duan_smearing_mean": float(np.mean(fold_smears)) if fold_smears else 1.0,
        "winsor_bounds": {"lo": float(clip_lo), "hi": float(clip_hi)},
        "stacker_used": bool(stacker_model is not None),
        "best_params": best_params,
        "metrics": metrics,
    }


def optimize_cogs_alpha(result_map: Dict[str, Dict[str, object]]) -> Tuple[float, Dict[str, float]]:
    cogs_oof = result_map["cogs"]["oof_frame"][[DATE_COL, "actual", "pred_final_base"]].rename(
        columns={"actual": "actual_cogs", "pred_final_base": "pred_cogs_direct"}
    )
    rev_oof = result_map["revenue"]["oof_frame"][[DATE_COL, "pred_final_base"]].rename(columns={"pred_final_base": "pred_revenue"})
    ratio_oof = result_map["cogs_ratio"]["oof_frame"][[DATE_COL, "pred_final_base"]].rename(columns={"pred_final_base": "pred_ratio"})
    merged = cogs_oof.merge(rev_oof, on=DATE_COL, how="inner").merge(ratio_oof, on=DATE_COL, how="inner")
    merged = merged.dropna(subset=["actual_cogs", "pred_cogs_direct", "pred_revenue", "pred_ratio"])
    if merged.empty:
        return 1.0, {"RMSE": np.nan, "MAE": np.nan}
    merged["pred_cogs_from_ratio"] = np.clip(merged["pred_revenue"] * merged["pred_ratio"], 0.0, None)
    y = merged["actual_cogs"].to_numpy(dtype=float)
    p_dir = merged["pred_cogs_direct"].to_numpy(dtype=float)
    p_rat = merged["pred_cogs_from_ratio"].to_numpy(dtype=float)
    best_alpha = 1.0
    best_rmse = float("inf")
    best_mae = float("inf")
    for alpha in np.arange(0.0, 1.0001, 0.01):
        pred = alpha * p_dir + (1.0 - alpha) * p_rat
        rmse = float(np.sqrt(mean_squared_error(y, pred)))
        if rmse < best_rmse:
            best_rmse = rmse
            best_alpha = float(alpha)
            best_mae = float(mean_absolute_error(y, pred))
    return best_alpha, {"RMSE": best_rmse, "MAE": best_mae}


def run_notebook_pipeline(data_dir: Optional[Path] = None, output_dir: Optional[Path] = None) -> Dict[str, str]:
    data_dir = find_data_dir(data_dir)
    output_dir = Path(output_dir) if output_dir is not None else Path("/kaggle/working/outputs")
    output_dir.mkdir(parents=True, exist_ok=True)

    sales = pd.read_csv(data_dir / "sales.csv")
    sales.columns = [c.strip().lower() for c in sales.columns]
    sales[DATE_COL] = pd.to_datetime(sales[DATE_COL], errors="coerce")
    sales = sales.dropna(subset=[DATE_COL]).sort_values(DATE_COL).reset_index(drop=True)
    sales["revenue"] = pd.to_numeric(sales["revenue"], errors="coerce").fillna(0.0)
    sales["cogs"] = pd.to_numeric(sales["cogs"], errors="coerce").fillna(0.0)
    sales["cogs_ratio"] = (sales["cogs"] / sales["revenue"].clip(lower=1.0)).clip(0.01, 3.0)

    sample_sub = pd.read_csv(data_dir / "sample_submission.csv")
    sub_date_col = "Date" if "Date" in sample_sub.columns else "date"
    horizon_dates = pd.DatetimeIndex(pd.to_datetime(sample_sub[sub_date_col], errors="coerce").dropna().sort_values().unique())
    CFG.val_days = min(CFG.val_days, len(horizon_dates))

    start_date = pd.Timestamp(sales[DATE_COL].min())
    end_date = pd.Timestamp(horizon_dates.max())
    det = build_deterministic_feature_frame(start_date, end_date)

    tables = {
        "orders": read_csv_if_exists(data_dir / "orders.csv", parse_dates=["order_date"], usecols=["order_id", "order_date", "customer_id", "order_status"]),
        "order_items": read_csv_if_exists(data_dir / "order_items.csv", usecols=["order_id", "product_id", "quantity", "unit_price", "discount_amount"]),
        "products": read_csv_if_exists(data_dir / "products.csv", usecols=["product_id", "category", "cogs"]),
        "promotions": read_csv_if_exists(data_dir / "promotions.csv", parse_dates=["start_date", "end_date"]),
        "inventory": read_csv_if_exists(data_dir / "inventory.csv", parse_dates=["snapshot_date"]),
        "web_traffic": read_csv_if_exists(data_dir / "web_traffic.csv", parse_dates=["date"]),
        "returns": read_csv_if_exists(data_dir / "returns.csv", parse_dates=["return_date"]),
        "reviews": read_csv_if_exists(data_dir / "reviews.csv", parse_dates=["review_date"]),
    }

    if CFG.use_cross_table_features:
        det = build_external_feature_store(det, tables)
    num_cols = [c for c in det.columns if pd.api.types.is_numeric_dtype(det[c])]
    det[num_cols] = det[num_cols].replace([np.inf, -np.inf], np.nan)

    gpu_count = torch.cuda.device_count()
    targets = ["revenue", "cogs", "cogs_ratio"]

    if gpu_count >= 2 and (not CFG.force_sequential):
        gpu_ids = list(range(gpu_count))
        n_jobs = min(2, len(targets), gpu_count)
        backend = "threading"
    else:
        gpu_ids = [0]
        n_jobs = 1
        backend = "sequential"

    jobs = [(t, gpu_ids[i % len(gpu_ids)], SEED + i) for i, t in enumerate(targets)]

    def one(job):
        t, gid, seed = job
        return train_target(t, gid, gpu_count, seed, sales, det, horizon_dates, output_dir)

    if n_jobs == 1:
        results = [one(job) for job in jobs]
    else:
        try:
            results = Parallel(n_jobs=n_jobs, backend=backend)(delayed(one)(job) for job in jobs)
        except Exception:
            results = [one(job) for job in jobs]

    result_map = {r["target"]: r for r in results}
    rev_fc = result_map["revenue"]["forecast"].reindex(horizon_dates)
    cogs_direct_fc = result_map["cogs"]["forecast"].reindex(horizon_dates)
    ratio_fc = result_map["cogs_ratio"]["forecast"].reindex(horizon_dates).clip(0.01, 3.0)
    cogs_from_ratio = np.clip(rev_fc.to_numpy(dtype=float) * ratio_fc.to_numpy(dtype=float), 0.0, None)
    alpha, alpha_stats = optimize_cogs_alpha(result_map)
    cogs_final = np.clip(alpha * cogs_direct_fc.to_numpy(dtype=float) + (1.0 - alpha) * cogs_from_ratio, 0.0, None)

    submission = pd.DataFrame({"Date": horizon_dates.strftime("%Y-%m-%d"), "Revenue": np.round(rev_fc.to_numpy(dtype=float), 2), "COGS": np.round(cogs_final, 2)})
    submission_path = output_dir / "submission_modeling.csv"
    submission.to_csv(submission_path, index=False)

    components = pd.DataFrame(
        {
            "Date": horizon_dates.strftime("%Y-%m-%d"),
            "Revenue_forecast": rev_fc.to_numpy(dtype=float),
            "COGS_direct_forecast": cogs_direct_fc.to_numpy(dtype=float),
            "COGS_ratio_forecast": ratio_fc.to_numpy(dtype=float),
            "COGS_from_ratio_forecast": cogs_from_ratio,
            "COGS_final_blend": cogs_final,
        }
    )
    components_path = output_dir / "forecast_components.csv"
    components.to_csv(components_path, index=False)

    summary = {
        "config": {
            "seed": SEED,
            "data_dir": str(data_dir),
            "output_dir": str(output_dir),
            "gpu_count": int(gpu_count),
            "mode": (f"parallel-{backend}" if n_jobs > 1 else "single-gpu/sequential"),
            "cv_max_splits": CFG.cv_max_splits,
            "initial_train_days": CFG.initial_train_days,
            "val_days": CFG.val_days,
            "step_days": CFG.step_days,
            "gap_days": CFG.gap_days,
            "enable_optuna": CFG.enable_optuna,
            "n_trials": CFG.n_trials,
            "optuna_timeout": CFG.optuna_timeout,
            "xgb_cv_estimators": CFG.xgb_cv_estimators,
            "xgb_final_estimators": CFG.xgb_final_estimators,
            "use_cross_table_features": CFG.use_cross_table_features,
            "train_recent_years": CFG.train_recent_years,
            "use_multi_seed": CFG.use_multi_seed,
            "bagging_seeds": CFG.bagging_seeds,
            "use_huber_model": CFG.use_huber_model,
            "huber_slope": CFG.huber_slope,
            "use_prophet_cv": CFG.use_prophet_cv,
            "use_prophet_final": CFG.use_prophet_final,
            "use_residual_sarima": CFG.use_residual_sarima,
            "use_residual_stacker": CFG.use_residual_stacker,
            "use_duan_smearing": CFG.use_duan_smearing,
            "use_linear_calibration": CFG.use_linear_calibration,
            "use_prediction_winsor": CFG.use_prediction_winsor,
            "winsor_q_low": CFG.winsor_q_low,
            "winsor_q_high": CFG.winsor_q_high,
            "composite_alpha": CFG.composite_alpha,
        },
        "targets": {},
        "cogs_hybrid": {"alpha_direct": alpha, "alpha_ratio": (1.0 - alpha), "oof_stats": alpha_stats},
        "outputs": {"submission": str(submission_path), "forecast_components": str(components_path)},
    }
    for t in targets:
        summary["targets"][t] = {
            "gpu_id": result_map[t]["gpu_id"],
            "best_params": result_map[t]["best_params"],
            "pred_cols": result_map[t]["pred_cols"],
            "blend_weights_global": result_map[t]["blend_weights_global"],
            "blend_weights_bucket": result_map[t]["blend_weights_bucket"],
            "event_multipliers": result_map[t]["event_multipliers"],
            "calibration": result_map[t]["calibration"],
            "duan_smearing_mean": result_map[t]["duan_smearing_mean"],
            "winsor_bounds": result_map[t]["winsor_bounds"],
            "stacker_used": result_map[t]["stacker_used"],
            "metrics": result_map[t]["metrics"],
        }

    summary_path = output_dir / "metrics_summary.json"
    with summary_path.open("w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    return {
        "submission": str(submission_path),
        "summary": str(summary_path),
        "components": str(components_path),
    }


In [ ]:
# Run full pipeline
outputs = run_notebook_pipeline()
outputs


In [ ]:
# Optional: zip artifacts
import zipfile
from pathlib import Path

out_dir = Path(outputs['submission']).parent
report_files = [
    out_dir / 'submission_modeling.csv',
    out_dir / 'metrics_summary.json',
    out_dir / 'forecast_components.csv',
    out_dir / 'oof_revenue.csv',
    out_dir / 'oof_cogs.csv',
    out_dir / 'oof_cogs_ratio.csv',
    out_dir / 'feature_importance_revenue.csv',
    out_dir / 'feature_importance_cogs.csv',
    out_dir / 'feature_importance_cogs_ratio.csv',
    out_dir / 'shap_revenue.csv',
    out_dir / 'shap_cogs.csv',
    out_dir / 'shap_cogs_ratio.csv',
]
zip_path = out_dir / 'report_bundle.zip'
with zipfile.ZipFile(zip_path, mode='w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in report_files:
        if p.exists():
            zf.write(p, arcname=p.name)
zip_path
